Scarping free patterns from https://www.crochet.com/patterns/free

In [2]:
import requests
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
import os

target_dir = "../frontend/src/assets/images"
os.makedirs(target_dir, exist_ok=True)

headers = {"User-Agent": "Mozilla/5.0"}

import cloudscraper
scraper = cloudscraper.create_scraper()

In [3]:
def scrape_product(product_link):
    row = {}
    os.makedirs(target_dir, exist_ok=True)
    scraper = cloudscraper.create_scraper()

    try:
        r = scraper.get(product_link)
        #r = requests.get(product_link, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"Error occurred while fetching {product_link}: {e}")
        return row

    # title
    title = soup.find("h1")
    if title:
        row["title"] = title.get_text(strip=True)
    else:
        row["title"] = "" 


    # find whether it is description or details button
    button = None
    for b in soup.find_all("button"):
        if "Pattern Description" in b.get_text():
            button = b
            desc_div = button.find_parent("h3").find_next_sibling("div")
            row["description"] = desc_div.get_text(" ", strip=True)
        
        if "Pattern Details" in b.get_text():
            button_details = b
            details_div = button_details.find_parent("h3").find_next_sibling("div")
            items = details_div.find_all("li")

            for li in items:
                label = li.find("b")

                if label:
                    key = label.get_text(strip=True).replace(":", "")
                    value = label.next_sibling.strip()

                    row[key] = value
    

    # image extraction and download
    img_container = soup.find("div", id = "img-zoom-stage")
    img_tag = img_container.select_one("img") if img_container else None

    img_url = None
    if img_tag:
        img_url = img_tag.get("src")
    row['image_url'] = img_url

    if img_url:
        safe_filename = re.sub(r'[^\w\s-]', '', row["title"]).strip().replace(' ', '_')
        save_path = os.path.join(target_dir, f"{safe_filename}.jpg")
        try:
            img_res = scraper.get(img_url, stream=True, timeout=10)
            if img_res.status_code == 200 and 'image' in img_res.headers.get('Content-Type', ''):
                with open(save_path, 'wb') as f:
                    for chunk in img_res.iter_content(1024):
                        f.write(chunk)
                row["local_path"] = f"{safe_filename}.jpg"
            else:
                print(f"Invalid image response for: {row['title']}")
        except Exception as e:
            print(f"Error downloading image for {row['title']}: {e}")
        
    return row

In [4]:
url = "https://www.crochet.com/filet-crochet-market-bag/p/58066"

scrape_product(url)

{'title': 'Filet Crochet Market Bagby Sara Dudek',
 'description': "Make a stunning market bag for trips to the farmers' market, outings to the beach, or to carry a few project bags worth of crochet goodies. You'll want to take this customizable bag everywhere! Practice the filet crochet technique with this mesh bag. Create patterns in the shape of a heart, a flower, a chevron pattern, or create a chart all your own. You will be a filet crochet expert by the time you finish this fast and fun market bag. Grab a project bundle and save!",
 'Pattern Type': 'Crochet',
 'Difficulty': 'Intermediate',
 'Sizes Included': '27" circumference, 15" long',
 'Yarn Weight': 'DK',
 'Yarn Line': '',
 'Yardage': '436',
 'Fiber Type': 'Cotton',
 'Needle / Hook Sizes': 'Size G (4.0mm) Crochet hook',
 'image_url': 'https://d2q9kw5vp0we94.cloudfront.net/i/_/big/58066220.jpg~w=1000,h=1000,v=1',
 'local_path': 'Filet_Crochet_Market_Bagby_Sara_Dudek.jpg'}

In [5]:
base_url = "https://www.crochet.com/patterns/free"
rows = []
page = 1

scraper = cloudscraper.create_scraper()

while True:
    print(f"Scraping page {page}...")

    if page == 1:
        url = base_url
    else:
        url = f"{base_url}?Free-Paid=Free&page={page}"

    if page == 8:
        break

    response = scraper.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.select("div.group.relative")
    if not cards:
        print("No more products found. Ending scrape.")
        break

    for card in cards:
        link = card.select_one("a[href]")
        product_link = "https://www.crochet.com" + link["href"] 

        # img_tag = card.select_one("img")
        # image_link = img_tag["src"] if img_tag else None

        # name_tag = card.select_one("h3 a")
        # name = name_tag.get_text(strip=True) if name_tag else None
        
        product_data = scrape_product(product_link)

        # product_data["name"] = name
        product_data["pattern_link"] = product_link
        # product_data["image_link"] = image_link

        rows.append(product_data)
        time.sleep(1)

    page += 1

print("Scraping completed. Total products scraped:", len(rows))
df = pd.DataFrame(rows)

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping completed. Total products scraped: 414


In [14]:
df.head()

,title,description,Pattern Type,Difficulty,Sizes Included,Yarn Weight,Yarn Line,Yardage,Fiber Type,Needle / Hook Sizes,image_url,local_path,pattern_link
0,Wavy Chevron Dishcloth,Save when you buy a project bundle! Striped or...,Crochet,Beginner,"Bright Colorway, Cabin Colorway, Candy Colorway",Worsted,,570,Cotton,Size G (4.0mm) crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Wavy_Chevron_Dishclothby_Jenny_Konopinski.jpg,https://www.crochet.com/wavy-chevron-dishcloth...
1,Filet Crochet Market Bag,Make a stunning market bag for trips to the fa...,Crochet,Intermediate,"27"" circumference, 15"" long",DK,,436,Cotton,Size G (4.0mm) Crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Filet_Crochet_Market_Bagby_Sara_Dudek.jpg,https://www.crochet.com/filet-crochet-market-b...
2,Good Earth Blanket,Get cozy in color with a striped crochet blank...,Crochet,Beginner,"35.75 x 48"" - spring picnic, 48.25 x 60"" - spr...",Worsted,,1744-2616,Acrylic,US 7 (4.5mm) crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Good_Earth_Blanketby_Pattern_Hut.jpg,https://www.crochet.com/good-earth-blanket/p/N...
3,Aberdeen Pullover,Bold and modern the geometric mosaic pattern o...,Crochet,Intermediate,"38.5"" finished chest, 42.75"" finished chest, 4...",Worsted,,2616-3924,Acrylic,H/5.0mm & I/5.5mm crochet hooks,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Aberdeen_Pulloverby_Moon_Elderidge.jpg,https://www.crochet.com/aberdeen-pullover/p/56291
4,Adventure Time Blanket,Playful vehicles set off for adventure on this...,Crochet,Intermediate,"24"" Wide x 36"" Long",Fingering,,1308,Cotton,US D/3 (3.25mm) Crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Adventure_Time_Blanketby_Amy_Polcyn_Niezur.jpg,https://www.crochet.com/adventure-time-blanket...


In [15]:
df['title'] = df['title'].str.replace(r"(?<=[a-z])by\s+.*$", "", regex=True)

In [16]:
df.head()

,title,description,Pattern Type,Difficulty,Sizes Included,Yarn Weight,Yarn Line,Yardage,Fiber Type,Needle / Hook Sizes,image_url,local_path,pattern_link
0,Wavy Chevron Dishcloth,Save when you buy a project bundle! Striped or...,Crochet,Beginner,"Bright Colorway, Cabin Colorway, Candy Colorway",Worsted,,570,Cotton,Size G (4.0mm) crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Wavy_Chevron_Dishclothby_Jenny_Konopinski.jpg,https://www.crochet.com/wavy-chevron-dishcloth...
1,Filet Crochet Market Bag,Make a stunning market bag for trips to the fa...,Crochet,Intermediate,"27"" circumference, 15"" long",DK,,436,Cotton,Size G (4.0mm) Crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Filet_Crochet_Market_Bagby_Sara_Dudek.jpg,https://www.crochet.com/filet-crochet-market-b...
2,Good Earth Blanket,Get cozy in color with a striped crochet blank...,Crochet,Beginner,"35.75 x 48"" - spring picnic, 48.25 x 60"" - spr...",Worsted,,1744-2616,Acrylic,US 7 (4.5mm) crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Good_Earth_Blanketby_Pattern_Hut.jpg,https://www.crochet.com/good-earth-blanket/p/N...
3,Aberdeen Pullover,Bold and modern the geometric mosaic pattern o...,Crochet,Intermediate,"38.5"" finished chest, 42.75"" finished chest, 4...",Worsted,,2616-3924,Acrylic,H/5.0mm & I/5.5mm crochet hooks,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Aberdeen_Pulloverby_Moon_Elderidge.jpg,https://www.crochet.com/aberdeen-pullover/p/56291
4,Adventure Time Blanket,Playful vehicles set off for adventure on this...,Crochet,Intermediate,"24"" Wide x 36"" Long",Fingering,,1308,Cotton,US D/3 (3.25mm) Crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Adventure_Time_Blanketby_Amy_Polcyn_Niezur.jpg,https://www.crochet.com/adventure-time-blanket...


In [17]:
df.loc[df['Difficulty'].isin(["Easy", "Beginner, Easy"]), 'Difficulty'] = "Beginner"

In [19]:
df[df.isna().any(axis=1)]

,title,description,Pattern Type,Difficulty,Sizes Included,Yarn Weight,Yarn Line,Yardage,Fiber Type,Needle / Hook Sizes,image_url,local_path,pattern_link
7,All About Shapes Ba,"A baby gift that is practical as well as cute,...",Crochet,Beginner,"Towel: 30"" square; Washcloth: 8” diameter",Worsted,,950,NaN,US G/7 (4.5 mm) crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,All_About_Shapes_Baby_Bath_Setby_Kristen_Clay.jpg,https://www.crochet.com/all-about-shapes-baby-...
10,Amigurumi Crochet Cupcake,"It's impossible to say ""no"" to a cupcake - esp...",NaN,NaN,One Size,Worsted,,200,NaN,E/3.5mm crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Amigurumi_Crochet_Cupcakeby_The_Itsy_Bitsy_Spi...,https://www.crochet.com/amigurumi-crochet-cupc...
15,Arwynn Blanket,Watch tranquil colors fade into each other as ...,Crochet,Beginner,"About 58 x 63""",Worsted,,3060,NaN,Size J/10 (6mm) crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Arwynn_Blanketby_Lion_Brand.jpg,https://www.crochet.com/arwynn-blanket/p/N8390
17,Aspen the Snowman,This pattern was inspired by the snow people w...,Crochet,Intermediate,"Approximately 9"" tall","Super Bulky, Worsted",,725,NaN,Sizes E/4 (3.50mm) & I/9 (5.50mm) Crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Aspen_the_Snowmanby_Ashley_Parker.jpg,https://www.crochet.com/aspen-the-snowman/p/58271
19,Aurora Wrap,Inspired by the interconnected geometric patte...,Crochet,Intermediate,"64.5 Wide x 18"" Long","Fingering, Lace",,3148,NaN,US E/6 (3.5 mm) and US D/3 (3mm): Crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Aurora_Wrapby_Jessica_Khochare.jpg,https://www.crochet.com/aurora-wrap/p/N5062
...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,Vista Wrap,Crochet your favorite vista into this meditati...,Crochet,Beginner,"70"" x 31""",Worsted,,1230,NaN,US H/8 (5mm) crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Vista_Wrapby_Two_of_Wands.jpg,https://www.crochet.com/vista-wrap/p/N8174
397,Waves Afghan,This easy afghan is fun and functional - a per...,Crochet,Beginner,"About 42 x 55"", including border",DK,,2320,NaN,US I-9/5.5mm crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Waves_Afghanby_Lion_Brand.jpg,https://www.crochet.com/waves-afghan/p/N8168
407,Winterberry Cowl,Indulge in instant warmth and style with this ...,Crochet,Beginner,22” Circumference x 10” Tall,Super Bulky,,208,NaN,US N/P-15 (10.0mm) Crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Winterberry_Cowlby_Jessica_Herr.jpg,https://www.crochet.com/winterberry-cowl/p/N5071
408,Wintergreen Chevron Hat,The subtle details in this fun hat are what ta...,NaN,Intermediate,"19.5"" hat circumference x 9.5"" tall, 22.75"" ha...",Worsted,,330-440,Wool,Sizes E/4 (3.5mm) & G (4.25mm) Crochet hook,https://d2q9kw5vp0we94.cloudfront.net/i/_/big/...,Wintergreen_Chevron_Hatby_Brenda_KB_Anderson.jpg,https://www.crochet.com/wintergreen-chevron-ha...


In [23]:
df['pattern_link'].nunique()

414

In [20]:
df['Difficulty'].value_counts()

Difficulty
Intermediate    264
Beginner        126
Advanced         19
Name: count, dtype: int64

In [21]:
df.iloc[10,1]

'It\'s impossible to say "no" to a cupcake - especially when these adorable sweets come with a 100 percent guilt free guarantee! This simple pattern is a great way to get started with amigurumi (the Japanese art of knitting or crocheting small stuffed animals and anthropomorphic creatures) because it uses basic shaping and can be customized over and over again. Before you know it, you\'ll be making these cuddly cupcakes for baby showers, office gifts, friends who could use a pick-me-up, and more. So let\'s get to baking... eh-hem, I mean crocheting shall we? Also, check out all of The Itsy Bitsy Spider IDP patterns right here.'

In [22]:
df.to_csv("../data/crochet_crochet_patterns.csv", index=False)